# Softmax — 2K Colab render

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lroe/softmax2/blob/main/render_2k_colab.ipynb)

This notebook renders the word-synced live-canvas explainer at **2560 × 1440** with headless Chromium and ffmpeg, then writes the MP4 to **Google Drive**.

- **Access first** — step 1 opens Google Drive's permission dialog before anything is written.
- **Resumable** — 600-frame chunks are encoded and copied to Drive. Rerun after a disconnect; finished chunks are skipped.
- **Live progress** — frame counts, chunk number, speed, and ETA stream into the bar in step 7.
- **No GPU** — Chromium and x264 run on CPU. A free Colab runtime works.
- **No skill file** — `skill.md` and `skill_original.md` are deleted from the runtime after clone and never copied to Drive.
- **Source** — https://github.com/lroe/softmax2

Expected runtime is roughly **1–3 hours** on a free Colab CPU runtime. Use `Runtime → Run all`, approve the Drive prompt, and let the render cell run. Interrupting is safe: completed chunks stay on Drive.


In [ ]:
# ---------------------------------------------------------------------------
# 1) Settings
# ---------------------------------------------------------------------------
from pathlib import Path

REPO_URL = "https://github.com/lroe/softmax2.git"
BRANCH = "main"

TARGET_WIDTH = 2560
TARGET_HEIGHT = 1440
FPS = 30
CHUNK_FRAMES = 600          # 20 s at 30 fps; copied to Drive after each chunk
CRF = 18                    # lower = better quality / larger file
OUTPUT_NAME = f"softmax-{TARGET_WIDTH}x{TARGET_HEIGHT}.mp4"
DRIVE_FOLDER = "Softmax2K"
STILL_TIME = 155.0          # seconds; the singers scene

SCALE = TARGET_WIDTH / 1920
assert TARGET_WIDTH % 2 == 0 and TARGET_HEIGHT % 2 == 0
assert abs(TARGET_HEIGHT - 1080 * SCALE) < 0.01, "Keep the 16:9 aspect ratio."

REPO_DIR = Path("/content/softmax2")
RENDER_DIR = REPO_DIR / "render"
WORK_DIR = Path("/content/softmax-render-work")

print(f"Target      : {TARGET_WIDTH} × {TARGET_HEIGHT} (scale {SCALE:.6f})")
print(f"Frame rate  : {FPS} fps")
print(f"Chunk size  : {CHUNK_FRAMES} frames")
print(f"CRF         : {CRF}")
print(f"Output name : {OUTPUT_NAME}")
print(f"Repo        : {REPO_DIR}")


## 1 · Google Drive access

This is the only place the notebook asks for permission. The next cell opens Google's authorization dialog. Nothing is written to Drive until you approve it.

Files will live in `MyDrive/<folder>/`:

```
Softmax2K/
├── softmax-2560x1440.mp4
├── chunks/                 # resumable 2K segments
├── stills/                 # 2K test frame
├── logs/                   # renderer event log
└── render_state.json       # completed chunk bookkeeping
```


In [ ]:
from google.colab import drive
from pathlib import Path
import json
import shutil

MOUNT = Path("/content/drive")
if not (MOUNT / "MyDrive").exists():
    print("Google Drive permission is required before rendering.")
    print("Approve the Google authorization popup, then return here.\n")
    drive.mount(str(MOUNT), force_remount=False)

assert (MOUNT / "MyDrive").exists(), "Google Drive was not mounted."

DRIVE_FOLDER = globals().get("DRIVE_FOLDER", "Softmax2K")
folder = input(f"Drive folder name [{DRIVE_FOLDER}]: ").strip()
if folder:
    DRIVE_FOLDER = folder
DRIVE_ROOT = MOUNT / "MyDrive" / DRIVE_FOLDER
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
for sub in ("chunks", "stills", "logs"):
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

probe = DRIVE_ROOT / ".write_test"
probe.write_text("ok\n")
probe.unlink()

(DRIVE_ROOT / "colab_render_config.json").write_text(json.dumps({
    "width": TARGET_WIDTH, "height": TARGET_HEIGHT, "fps": FPS,
    "chunk_frames": CHUNK_FRAMES, "crf": CRF, "output": OUTPUT_NAME,
}, indent=2) + "\n")

print("\nDrive access granted.")
print(f"Output folder: {DRIVE_ROOT}")
try:
    free = shutil.disk_usage(DRIVE_ROOT).free / 1e9
    print(f"Drive free   : {free:.1f} GB")
except OSError:
    pass


## 2 · Install Node, ffmpeg and Chromium libraries

Colab usually has ffmpeg. This cell adds Node 22 if needed and the shared libraries that Playwright's Chromium expects. It can take a few minutes on the first run.


In [ ]:
import os
import subprocess


def run_stream(cmd, cwd=None, check=True, env=None):
    """Run a command and print every line as it arrives."""
    printable = " ".join(str(x) for x in cmd)
    print(f"$ {printable}", flush=True)
    merged_env = {**os.environ, "DEBIAN_FRONTEND": "noninteractive", **(env or {})}
    proc = subprocess.Popen(
        [str(x) for x in cmd],
        cwd=cwd,
        env=merged_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    code = proc.wait()
    if check and code != 0:
        raise RuntimeError(f"Command failed with exit code {code}: {printable}")
    return code


def node_major():
    try:
        raw = subprocess.check_output(["node", "--version"], text=True).strip()
        return int(raw.lstrip("v").split(".")[0])
    except Exception:
        return 0


print("Updating apt and installing the base tools...")
run_stream(["apt-get", "update", "-qq"])
run_stream([
    "apt-get", "install", "-y", "-qq",
    "ffmpeg", "git", "curl", "ca-certificates",
    "fonts-dejavu-core", "fonts-liberation",
])

if node_major() < 20:
    print("\nInstalling Node.js 22...")
    run_stream(["bash", "-lc", "curl -fsSL https://deb.nodesource.com/setup_22.x | bash -"])
    run_stream(["apt-get", "install", "-y", "-qq", "nodejs"])
    for name in ("node", "npm", "npx"):
        target = Path("/usr/bin") / name
        if target.exists():
            try:
                (Path("/usr/local/bin") / name).unlink(missing_ok=True)
                (Path("/usr/local/bin") / name).symlink_to(target)
            except OSError:
                pass

print("\nVersions:")
run_stream(["node", "--version"])
run_stream(["npm", "--version"])
run_stream(["ffmpeg", "-version"])


## 3 · Clone or update the project

Always checks out the latest `main`. A fresh clone includes the narration WAV, so the first clone can take a moment.

The production playbook (`skill.md` / `skill_original.md`) is removed from the runtime after clone. It is never copied to Drive.


In [ ]:
import shutil
import subprocess

print("Fetching the latest project from GitHub...")
if (REPO_DIR / ".git").exists():
    run_stream(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", BRANCH])
    run_stream(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"])
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    run_stream([
        "git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)
    ])

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "log", "-1", "--format=%h  %ad  %s", "--date=iso"],
    text=True,
).strip()
print("\nChecked out:", commit)

removed = []
for name in ("skill.md", "skill_original.md"):
    p = REPO_DIR / name
    if p.exists():
        p.unlink()
        removed.append(name)
print("Removed from runtime (not uploaded):", ", ".join(removed) if removed else "none")

assert (REPO_DIR / "preview" / "index.html").exists(), "preview/index.html is missing."
assert (RENDER_DIR / "render-2k.mjs").exists(), "render/render-2k.mjs is missing."
audio = REPO_DIR / "audio.wav"
assert audio.exists(), "audio.wav is missing."
print(f"Narration: {audio.stat().st_size / 1e6:.1f} MB")


## 4 · Install the browser and Playwright

`npm ci` installs the exact Playwright version from `package-lock.json`. The cell remembers the lockfile hash, so rerunning it in the same runtime is instant. `playwright install chromium --with-deps` downloads the headless browser.


In [ ]:
import hashlib

lock = RENDER_DIR / "package-lock.json"
stamp = RENDER_DIR / ".install-stamp"
node_version = subprocess.check_output(["node", "--version"], text=True).strip()
expected = hashlib.sha256(lock.read_bytes()).hexdigest() + "|" + node_version

if stamp.exists() and stamp.read_text().strip() == expected:
    print("node_modules already matches package-lock.json.")
else:
    run_stream(["npm", "ci", "--no-audit", "--no-fund"], cwd=RENDER_DIR)
    stamp.write_text(expected + "\n")

playwright_version = subprocess.check_output(
    ["node", "-e", "console.log(require('playwright/package.json').version)"],
    cwd=RENDER_DIR,
    text=True,
).strip()
print(f"Playwright {playwright_version}")

print("\nEnsuring Chromium is installed...")
run_stream(["npx", "playwright", "install", "chromium", "--with-deps"], cwd=RENDER_DIR)
print("Browser ready.")


## 5 · Verify a 2K frame before the long render

This renders the singers scene at 2560 × 1440, checks the pixel dimensions, saves it to Drive, and shows it inline. If this frame looks right, the full render will too.


In [ ]:
import shutil
from PIL import Image
from IPython.display import Image as IPImage, display

WORK_DIR.mkdir(parents=True, exist_ok=True)
still_local = WORK_DIR / f"still_{int(STILL_TIME):04d}.png"
still_drive = DRIVE_ROOT / "stills" / f"frame_{int(STILL_TIME):04d}_{TARGET_WIDTH}x{TARGET_HEIGHT}.png"

print(f"Rendering still at {STILL_TIME:.1f}s and {TARGET_WIDTH} × {TARGET_HEIGHT}...")
run_stream([
    "node", "render/render-2k.mjs",
    "--still", str(STILL_TIME),
    "--still-out", str(still_local),
    "--width", str(TARGET_WIDTH),
    "--height", str(TARGET_HEIGHT),
], cwd=REPO_DIR)

im = Image.open(still_local)
print(f"\nRendered size: {im.width} × {im.height}")
assert (im.width, im.height) == (TARGET_WIDTH, TARGET_HEIGHT), "The 2K still has the wrong dimensions."

shutil.copy2(still_local, still_drive)
print(f"Saved to Drive: {still_drive}")
display(IPImage(filename=str(still_drive), width=960))


## 6 · Render the full video with live progress

The renderer splits the timeline into 600-frame chunks, renders each one at 2K, encodes it with x264, and copies it to Drive before starting the next. The progress bar tracks all frames; the status text shows the current chunk, speed, and ETA.

- **Stop safely:** interrupt this cell. Completed chunks stay on Drive.
- **Resume:** rerun from the top. Finished chunks are skipped automatically.
- **Final file:** stitched from the chunks and the narration WAV.


In [ ]:
import json
import shutil
import subprocess
import time
from datetime import datetime

from IPython.display import FileLink, display
from tqdm.auto import tqdm

WORK_DIR.mkdir(parents=True, exist_ok=True)
log_local = WORK_DIR / f"render_{datetime.now():%Y%m%d_%H%M%S}.log"
log_drive = DRIVE_ROOT / "logs" / "render.log"
log_handle = open(log_local, "w", encoding="utf-8")

def fmt_eta(seconds):
    seconds = max(0, int(seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f"{h}h {m:02d}m" if h else f"{m}m {s:02d}s"

cmd = [
    "node", "render/render-2k.mjs",
    "--output-dir", str(DRIVE_ROOT),
    "--work-dir", str(WORK_DIR),
    "--output-name", OUTPUT_NAME,
    "--chunk", str(CHUNK_FRAMES),
    "--fps", str(FPS),
    "--crf", str(CRF),
    "--width", str(TARGET_WIDTH),
    "--height", str(TARGET_HEIGHT),
]

print("$", " ".join(cmd))
print("Finished chunks are copied to Google Drive before the next one starts.\n")

bar = tqdm(total=0, unit="frame", smoothing=0.05, dynamic_ncols=True)
proc = None
last_status = 0.0

try:
    proc = subprocess.Popen(
        cmd,
        cwd=REPO_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for raw in proc.stdout:
        line = raw.rstrip("\n")
        log_handle.write(line + "\n")
        log_handle.flush()

        if not line.startswith("@@RENDER@@ "):
            if line.strip():
                print(line, flush=True)
            continue

        event = json.loads(line[len("@@RENDER@@ "):])
        kind = event.get("type")

        if kind == "start":
            bar.total = int(event["totalFrames"])
            bar.refresh()
            print(
                f"Rendering {event['totalFrames']:,} frames in "
                f"{event['chunkCount']} chunks at {event['width']} × {event['height']}.",
                flush=True,
            )
        elif kind == "chunk_start":
            print(
                f"[chunk {event['chunk']}/{event['chunkCount']}] "
                f"frames {event['startFrame']:,}–{event['endFrame']:,}",
                flush=True,
            )
        elif kind == "chunk_skip":
            bar.update(int(event.get("frames", 0)))
            print(f"[chunk {event['chunk']}/{event['chunkCount']}] already on Drive; skipped.", flush=True)
        elif kind == "progress":
            bar.n = int(event["frame"])
            bar.set_postfix(
                fps=f"{event['fps']:.1f}",
                eta=fmt_eta(event["etaSeconds"]),
                chunk=f"{event['chunk']}/{event['chunkCount']}",
            )
            bar.refresh()
            if time.time() - last_status >= 5:
                print(
                    f"[{event['percent']:5.1f}%] "
                    f"{event['frame']:,}/{event['totalFrames']:,} frames | "
                    f"{event['fps']:.2f} fps | ETA {fmt_eta(event['etaSeconds'])}",
                    flush=True,
                )
                last_status = time.time()
                try:
                    shutil.copy2(log_local, log_drive)
                except OSError:
                    pass
        elif kind == "chunk_done":
            mb = event["bytes"] / 1e6
            print(
                f"[chunk {event['chunk']}/{event['chunkCount']}] done: "
                f"{mb:.1f} MB in {event['seconds']:.1f}s → {event['file']}",
                flush=True,
            )
            try:
                shutil.copy2(log_local, log_drive)
            except OSError:
                pass
        elif kind == "final":
            print(f"\nFINAL: {event['file']} ({event['bytes'] / 1e9:.2f} GB)", flush=True)
            print(event.get("probe", ""), flush=True)
        elif kind == "error":
            print("\nRENDER ERROR:\n" + event.get("message", ""), flush=True)

    code = proc.wait()
    if code != 0:
        tail = log_local.read_text(errors="replace")[-4000:]
        raise RuntimeError(f"Renderer exited with code {code}.\n{tail}")

except KeyboardInterrupt:
    print("\nInterrupted. Completed chunks are safe on Drive.")
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=20)
        except subprocess.TimeoutExpired:
            proc.kill()
    print("Rerun this cell to resume; finished chunks are skipped.")
finally:
    bar.close()
    log_handle.close()
    try:
        shutil.copy2(log_local, log_drive)
    except OSError:
        pass

final = DRIVE_ROOT / OUTPUT_NAME
if final.exists():
    print(f"\nFinal file: {final}")
    display(FileLink(str(final)))


## 7 · Verify the final file

The MP4 should be 2560 × 1440, about 19:55 long, with the narration track.


In [ ]:
import subprocess
from IPython.display import FileLink, display

final = DRIVE_ROOT / OUTPUT_NAME
if not final.exists():
    print("Final file not found yet. Run the render cell first.")
else:
    print(f"Final file : {final}")
    print(f"Size       : {final.stat().st_size / 1e9:.2f} GB")
    probe = subprocess.check_output([
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration,size",
        "-show_entries", "stream=codec_type,width,height,r_frame_rate",
        "-of", "default=noprint_wrappers=1",
        str(final),
    ], text=True)
    print(probe.strip())
    display(FileLink(str(final)))


## 8 · Optional housekeeping

The final MP4 is independent of the chunk files. Once you have downloaded and played the final file, you can delete `chunks/` to free Drive space. Leave `DELETE_CHUNKS = False` until then.


In [ ]:
import shutil

DELETE_CHUNKS = False

final = DRIVE_ROOT / OUTPUT_NAME
chunks = sorted((DRIVE_ROOT / "chunks").glob("chunk_*.mp4"))

if DELETE_CHUNKS and final.exists():
    freed = sum(p.stat().st_size for p in chunks)
    for p in chunks:
        p.unlink()
    print(f"Deleted {len(chunks)} chunks ({freed / 1e9:.2f} GB freed).")
elif DELETE_CHUNKS:
    print("Refusing to delete chunks because the final MP4 is missing.")
else:
    total = sum(p.stat().st_size for p in chunks)
    print(f"Kept {len(chunks)} chunks ({total / 1e9:.2f} GB). Set DELETE_CHUNKS = True after verifying the final file.")


## Troubleshooting

| Symptom | Fix |
| --- | --- |
| `Google Drive was not mounted` | Rerun step 1 and approve the popup. Popup blockers can stop it; allow popups for `colab.research.google.com`. |
| Chromium will not start | Rerun step 2, then step 4. The 2K still in step 5 is the quickest test. |
| Render stops when Colab disconnects | Rerun from the top. Completed chunks and `render_state.json` are on Drive and are skipped. |
| Drive runs out of space | Set `DELETE_CHUNKS = True` after the final MP4 is verified; or raise `CRF` for a smaller file. |
| The final file is missing but chunks exist | Run the render cell again; the renderer will skip finished chunks and stitch. |
| Want a smaller file | Raise `CRF` in step 1 (for example 20–22) and use `--force` only if you intend to re-render every chunk. |

The renderer is `render/render-2k.mjs`. It drives the existing preview with Playwright, captures the 2560 × 1440 canvas, encodes resumable chunks, and muxes the narration.
